# Structured Outputs

*Notebook 05*

Free-form text is flexible for people and fragile for parsers.

Structured outputs give downstream code a validated type.

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner
from typing import Optional, Annotated, Literal
from pydantic import BaseModel, Field, ValidationError, model_validator
from datetime import date

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

Humans can interpret flexible prose.

Code needs stable fields and types.

In [ ]:
# Free-form text is readable for humans but fragile for field access
unstructured_agent = Agent(
    name="UnstructuredAnalyst",
    instructions="Analyze the sentiment of the given text. Respond with your analysis.",
    model=MODEL
)

review = (
    "The product arrived on time and works great. "
    "Very happy with this purchase."
)

result = await Runner.run(unstructured_agent, input=review)

print("Raw output:")
print(result.final_output)
print()

# Try to use the output as structured data (it's still a string, so this fails)
output = result.final_output
try:
    sentiment = output["sentiment"]  # TypeError: string indices must be integers
except TypeError:
    print("❌ result.final_output is a string, there's no field to access")

### 💡 Why This Breaks

Without `output_type`, `result.final_output` is a plain string.

---

## 📐 Part 1: Defining Output Structure with Pydantic

Define a Pydantic model and pass it through `output_type=`.

The SDK derives a JSON schema and requests structured output.

On success, `result.final_output` is a validated instance of your model.

Schema-invalid output raises instead of reaching downstream code.

Use `Literal[...]` when a field must contain one of a fixed set of values.

In [ ]:
class SentimentResult(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: Annotated[float, Field(ge=0.0, le=1.0)]
    key_phrase: str
    reasoning: str


# --------------------------------------------------------------
print("✅ SentimentResult model defined")
print(f"    Fields: {list(SentimentResult.model_fields.keys())}")

#### Run with Structured Output

In [ ]:
sentiment_instructions = (
    "Analyze the sentiment of the given text.\n"
    "Classify as positive, negative, or neutral.\n"
    "Identify the key phrase that most signals the sentiment.\n"
    "Provide a confidence score from 0.0 to 1.0."
)

sentiment_agent = Agent(
    name="SentimentAnalyst",
    instructions=sentiment_instructions,
    model=MODEL,
    output_type=SentimentResult
)

review = (
    "The product arrived on time and works great. "
    "Very happy with this purchase."
)

result = await Runner.run(sentiment_agent, input=review)

analysis = result.final_output

print(f"Sentiment:    {analysis.sentiment}")
print(f"Confidence:  {analysis.confidence:.0%}")
print(f"Key phrase:  '{analysis.key_phrase}'")
print(f"Reasoning:    {analysis.reasoning}")

### 💡 Key Takeaway

A successful structured run guarantees the schema, not factual truth.

Validate important facts separately.

---

## 🧩 Part 2: Optional Values

Some values may be unavailable.

Use `Optional[type] = None` to represent either a value or `None`.

In the SDK's strict schema, the field remains present but may be `null`.

In [ ]:
class ContactInfo(BaseModel):
    name: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    company: Optional[str] = None


extractor_instructions = (
    "Extract contact information from the given text.\n"
    "Only populate fields that are clearly present in the text."
)

extractor_agent = Agent(
    name="ContactExtractor",
    instructions=extractor_instructions,
    model=MODEL,
    output_type=ContactInfo
)

# --------------------------------------------------------------
print("✅ ContactExtractor agent ready")

#### Extract from Multiple Texts

In [ ]:
texts = [
    "Please reach out to Sarah Chen at sarah@techcorp.com or call 555-0142.",
    "Contact Marcus for details.",
    "Email hello@startup.io to schedule a demo with our sales team."
]

print("📋 CONTACT EXTRACTION")
print("=" * 60)

for text in texts:
    result = await Runner.run(extractor_agent, input=text)

    contact = result.final_output

    is_complete = contact.name is not None and bool(contact.email or contact.phone)

    display_text = f"{text[:50]}..." if len(text) > 50 else text
    print(f"\nText: \"{display_text}\"")
    print(f"  Name:      {contact.name or '-'}")
    print(f"  Email:     {contact.email or '-'}")
    print(f"  Phone:     {contact.phone or '-'}")
    print(f"  Complete: {'✅' if is_complete else '⚠️  Incomplete'}")

### 💡 Key Takeaway

Optional values make absence explicit.

Downstream code can test for `None` directly.

---

## 🧪 Part 3: Semantic Validation

JSON schema handles field types and supported constraints.

Some rules depend on relationships between fields:

- Conditional requirements

- Cross-field relationships

- Date ordering

Use Python validators for those rules.

In [ ]:
class DateRange(BaseModel):
    event_name: str
    start_date: date
    end_date: date

    @model_validator(mode="after")
    def end_after_start(self):
        if self.end_date <= self.start_date:
            raise ValueError(
                f"end_date ({self.end_date}) must be after "
                f"start_date ({self.start_date})"
            )
        return self


# --------------------------------------------------------------
print("✅ DateRange model defined")

#### Valid and Invalid Examples

In [ ]:
# Valid: end after start
valid_range = DateRange(
    event_name="Team Offsite",
    start_date=date(2026, 5, 10),
    end_date=date(2026, 5, 12)
)
print(f"✅ Valid: {valid_range.event_name} ({valid_range.start_date} → {valid_range.end_date})")

# Invalid: end before start
print()
try:
    DateRange(
        event_name="Broken Event",
        start_date=date(2026, 5, 12),
        end_date=date(2026, 5, 10)
    )
except ValidationError as error:
    print(f"❌ {error.errors()[0]['msg']}")

### 💡 Key Takeaway

`@model_validator` runs when Pydantic validates the result.

Its Python logic is not included in the schema sent to the model.

State important semantic rules in the instructions, then validate them in code.

---

## 💪 Practice Exercises

### Exercise 1: Meeting Notes Extractor

*Covers: `output_type`, extracting structured data from unstructured text*

Build an agent that extracts structured information from raw meeting notes.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Meeting Notes Extractor
# --------------------------------------------------------------
# Objective: Extract structured data from unstructured meeting notes.

meeting_notes = """
Weekly sync, Jan 15
Attendees: Priya, Tom, Leila
We decided to launch the beta on Feb 1st. Tom will handle the
infrastructure setup by Jan 25. Priya needs to finish the onboarding
flow before that. Leila is out next week so the design review moves to Jan 22.
"""

# TODO 1: Define an ActionItem model with fields:
#            - person (str)
#            - task (str)
#            - due_date (str)

# TODO 2: Define a MeetingNotes model with fields for:
#            - date (str)
#            - attendees (list[str])
#            - decisions (list[str])
#            - action_items (list[ActionItem])
#            - next_meeting (Optional[str] = None)

# TODO 3: Create an agent with output_type=MeetingNotes
#            Instructions: extract all structured information from meeting notes

# TODO 4: Run the agent on meeting_notes
#         Assign result.final_output to meeting

# TODO 5: Print each field, and show typed nested access:
#            meeting.action_items[0].person, meeting.attendees[0], etc.
#            (no string parsing)

# --- Write your code below this line ---

### Exercise 2: Content Classifier

*Covers: `output_type`, classification with structured reasoning*

Build an agent that classifies text content and explains its reasoning.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Content Classifier
# --------------------------------------------------------------
# Objective: Classify content and confirm downstream code can use the result.

samples = [
    "The new onboarding flow reduced user drop-off by 40% in the first week.",
    "We need to reschedule the Q3 planning meeting. Friday no longer works for the team.",
    "The API documentation is missing examples for the authentication endpoints."
]

# TODO 1: Define a ContentClassification Pydantic model with:
#            - category (Literal["metrics", "scheduling", "documentation", "support", "other"])
#            - subcategory (str)   (more specific label)
#            - confidence (Annotated[float, Field(ge=0.0, le=1.0)])
#            - reasoning (str)     (one sentence explanation)

# TODO 2: Create a classifier agent with output_type=ContentClassification

# TODO 3: Run the agent on each sample in a loop

# TODO 4: Group results by category and print a summary
#            e.g. "metrics: 1 sample, scheduling: 1 sample, documentation: 1 sample"
#            This only works cleanly because the output is structured

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**`output_type=` returns typed data on successful runs:**

- The SDK derives a schema from a supported Python output type

- `result.final_output` is a validated object, not a string

- Schema validation checks structure, not factual correctness
<br>
<br>

**Structured outputs support downstream code:**

- Access fields directly without parsing prose

- Aggregate, filter, or group validated fields

- Successful runs use a stable field structure
<br>
<br>

**Design models for their consumers:**

- Use `Optional[type] = None` for nullable values

- Keep field names clear and models minimal

- Prefer structured output when downstream code needs fields
<br>
<br>

**Semantic validators run in Python:**

- `@model_validator` can enforce relationships between fields

- Its logic is not part of the model-facing JSON schema

- Failed validation surfaces before downstream code uses the result

---

## 📍 Next Step

**Notebook 06: Error Handling & Recovery**  

See how tool failures surface.

Then add fallbacks and retries.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-05-structured-outputs)

---